# 단계 12. AlphaEarth 분류모델 3종 (산지 피복 LC · 임상 FT · 변화유형 CH)

논문 3.7절 · 절차서: `forest-change-agent/EXPERIMENTS.md` 순서 12

입력: `output/ae_class_samples.npz`(`prep/build_ae_class_samples.py`), 단계 01 AE 래스터

산출: Drive `GEE_Export/ae_class_models` → 로컬 `ae_class_models/` → `forest_decision_agent/build_ae_classes.py` 에서 적용

> 셀 출력은 원 실행 기록입니다(원본 노트북: scratchpad (6).ipynb). 당시 이름(Mini-GFM, `mini_gfm_ts.py`, `GEE_Export/mini_gfm_chuncheon` 등)은 코드에서 S2-TSFM·단계 번호 이름으로 바꿨고, 출력 문구는 실행 당시 그대로입니다.


In [1]:
# [12-1] Drive 연결 + AlphaEarth 2017~2025 파일 확인
from google.colab import drive
drive.mount('/content/drive')
import glob, os
AE = {}
for y in range(2017, 2026):
    f = sorted(glob.glob(f'/content/drive/MyDrive/**/Chuncheon_AE_{y}_64Bands_int8*.tif', recursive=True))
    AE[y] = f[0] if f else None
    print(y, AE[y], round(os.path.getsize(f[0]) / 1e6) if f else '')

Mounted at /content/drive
2017 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2017_64Bands_int8.tif 925
2018 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2018_64Bands_int8.tif 926
2019 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2019_64Bands_int8.tif 922
2020 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2020_64Bands_int8.tif 926
2021 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2021_64Bands_int8.tif 929
2022 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2022_64Bands_int8.tif 918
2023 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2023_64Bands_int8.tif 925
2024 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2024_64Bands_int8.tif 925
2025 /content/drive/MyDrive/GEE_Export/Chuncheon_AE_2025_64Bands_int8.tif 924


In [2]:
# [12-2] 표본 업로드: C:\src\ForestChangeGeoAI\output\ae_class_samples.npz 선택 (Drive 사본이 있으면 생략)
import shutil, numpy as np
from google.colab import files
os.makedirs('/content/data', exist_ok=True)
G = '/content/drive/MyDrive/GEE_Export/ae_class_samples.npz'
dst = '/content/data/ae_class_samples.npz'
if not os.path.exists(dst):
    for fn in files.upload():
        shutil.move(fn, dst)
    shutil.copy(dst, G)
S = dict(np.load(dst))
print({k: v.shape for k, v in S.items()})
import collections; print(collections.Counter(zip(S['task'], S['year'], S['label'])))

Saving ae_class_samples.npz to ae_class_samples.npz
{'task': (60698,), 'year': (60698,), 'label': (60698,), 'poly': (60698,), 'row': (60698,), 'col': (60698,), 'test': (60698,)}
Counter({(np.str_('FT'), np.int16(2019), np.str_('침엽수림')): 6000, (np.str_('FT'), np.int16(2019), np.str_('활엽수림')): 6000, (np.str_('FT'), np.int16(2025), np.str_('침엽수림')): 6000, (np.str_('FT'), np.int16(2025), np.str_('활엽수림')): 6000, (np.str_('CH'), np.int16(2025), np.str_('무변화')): 6000, (np.str_('FT'), np.int16(2025), np.str_('혼효림')): 5964, (np.str_('FT'), np.int16(2019), np.str_('혼효림')): 5739, (np.str_('CH'), np.int16(2025), np.str_('벌채·갱신')): 4318, (np.str_('LC'), np.int16(2019), np.str_('산림')): 3000, (np.str_('LC'), np.int16(2025), np.str_('산림')): 3000, (np.str_('CH'), np.int16(2025), np.str_('산림증가')): 2840, (np.str_('LC'), np.int16(2025), np.str_('시설지')): 1719, (np.str_('LC'), np.int16(2019), np.str_('시설지')): 1399, (np.str_('LC'), np.int16(2019), np.str_('나지')): 1080, (np.str_('LC'), np.int16(2025), np.str_

In [3]:
# [12-3] 표본 화소의 AlphaEarth 임베딩 추출 (2017~2025, 64차원, int8/127 → 단위벡터)
import rasterio, time
os.makedirs('/content/ae', exist_ok=True)
r, c = S['row'].astype(int), S['col'].astype(int)
EMB = {}
t0 = time.time()
for y in range(2017, 2026):
    loc = f'/content/ae/AE_{y}.tif'
    if not os.path.exists(loc):
        shutil.copy(AE[y], loc)
    with rasterio.open(loc) as ds:
        a = ds.read()                                   # (64, H, W) int8
    v = a[:, r, c].T.astype('float32')
    del a
    bad = (v == -128).any(1) | (v == 0).all(1)
    v = v / 127.0
    v /= np.clip(np.linalg.norm(v, axis=1, keepdims=True), 1e-6, None)
    v[bad] = np.nan
    EMB[y] = v
    print(y, f'결측 {bad.mean():.3%}', f'{time.time()-t0:.0f}s', flush=True)
np.savez_compressed('/content/drive/MyDrive/GEE_Export/ae_class_sample_embeddings.npz', **{str(k): v for k, v in EMB.items()})

2017 결측 0.000% 40s
2018 결측 0.000% 73s
2019 결측 0.000% 106s
2020 결측 0.000% 134s
2021 결측 0.000% 168s
2022 결측 0.000% 201s
2023 결측 0.000% 226s
2024 결측 0.000% 260s
2025 결측 0.000% 290s


In [4]:
# [12-4] 세 분류 모델 학습·독립 블록 검증 (HistGradientBoosting, 클래스 균형) → Drive/GEE_Export/ae_class_models
import json, joblib, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
YRS = list(range(2017, 2026))
ALLMEAN = np.nanmean(np.stack([EMB[y] for y in YRS]), 0)

def state_feats(idx, years, multi=True):
    """상태 분류(LC·FT): 해당 연도 임베딩 (+ 앞·뒤 연도, 2017~2025 평균)"""
    out = np.full((len(idx), 256 if multi else 64), np.nan, 'float32')
    for y in np.unique(years):
        m = years == y
        ii = idx[m]
        f = [EMB[y][ii]]
        if multi:
            f += [EMB[max(y - 1, 2017)][ii], EMB[min(y + 1, 2025)][ii], ALLMEAN[ii]]
        out[m] = np.concatenate(f, 1)
    return out

def change_feats(idx):
    e19, e25 = EMB[2019][idx], EMB[2025][idx]
    cons = np.stack([1 - (EMB[y - 1][idx] * EMB[y][idx]).sum(1) for y in range(2018, 2026)], 1)   # 2017→18 … 2024→25
    win = cons[:, 2:]                                                                              # 2019→20 … 2024→25
    return np.concatenate([e19, e25, e25 - e19, (1 - (e19 * e25).sum(1))[:, None], cons,
                           win.max(1, keepdims=True), (2020 + np.nanargmax(np.nan_to_num(win, nan=-1), 1))[:, None],
                           cons[:, :2].max(1, keepdims=True)], 1).astype('float32')

def model():
    return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=1.0,
                                          class_weight='balanced', random_state=42)

OUTD = '/content/drive/MyDrive/GEE_Export/ae_class_models'; os.makedirs(OUTD, exist_ok=True)
TASKS = {'LC': ('산지 피복: 산림·나지·초지·시설지', ['산림', '나지', '초지', '시설지']),
         'FT': ('임상: 침엽수림·활엽수림·혼효림', ['침엽수림', '활엽수림', '혼효림']),
         'CH': ('변화유형 2019→2025: 무변화·벌채·갱신·산림증가 (산지전용 제외)', ['무변화', '벌채·갱신', '산림증가'])}
report = {}
for t, (desc, labels) in TASKS.items():
    sel = np.flatnonzero(S['task'] == t)
    y = S['label'][sel]; yr = S['year'][sel]; te = S['test'][sel] == 1
    variants = {'다년(2017~2025)': change_feats(sel) if t == 'CH' else state_feats(sel, yr, True)}
    if t != 'CH':
        variants['단일연도'] = state_feats(sel, yr, False)
    else:
        variants['두 시점(2019·2025)만'] = np.concatenate([EMB[2019][sel], EMB[2025][sel], EMB[2025][sel] - EMB[2019][sel]], 1)
    report[t] = {'과업': desc, '표본': {'학습': int((~te).sum()), '검증': int(te.sum())},
                 '클래스별_표본': {l: [int(((y == l) & ~te).sum()), int(((y == l) & te).sum())] for l in labels}}
    for vn, X in variants.items():
        ok = ~np.isnan(X).all(1)
        m = model().fit(X[~te & ok], y[~te & ok])
        p = m.predict(X[te & ok])
        cr = classification_report(y[te & ok], p, labels=labels, output_dict=True, zero_division=0)
        report[t][vn] = {'macro_F1': round(cr['macro avg']['f1-score'], 3), '정확도': round(cr['accuracy'], 3),
                         '클래스별': {l: {k: round(cr[l][k], 3) for k in ('precision', 'recall', 'f1-score')} for l in labels},
                         '혼동행렬(행=실제)': confusion_matrix(y[te & ok], p, labels=labels).tolist(), '특징수': X.shape[1]}
        print(f'[{t}] {vn:<16} 특징 {X.shape[1]:>3} | macro-F1 {report[t][vn]["macro_F1"]:.3f} 정확도 {report[t][vn]["정확도"]:.3f} | '
              + ', '.join(f'{l} {cr[l]["f1-score"]:.2f}' for l in labels), flush=True)
        if vn.startswith('다년'):
            final = model().fit(X[ok], y[ok])                    # 지도 적용용: 전체 표본으로 재학습
            joblib.dump({'model': final, 'labels': labels, 'features': '다년' if t != 'CH' else 'change'}, f'{OUTD}/{t}_model.joblib')
json.dump(report, open(f'{OUTD}/report.json', 'w'), ensure_ascii=False, indent=1)
print('저장:', OUTD)

[LC] 다년(2017~2025)    특징 256 | macro-F1 0.622 정확도 0.835 | 산림 0.92, 나지 0.64, 초지 0.11, 시설지 0.82
[LC] 단일연도             특징  64 | macro-F1 0.633 정확도 0.823 | 산림 0.92, 나지 0.64, 초지 0.19, 시설지 0.79
[FT] 다년(2017~2025)    특징 256 | macro-F1 0.646 정확도 0.647 | 침엽수림 0.67, 활엽수림 0.70, 혼효림 0.57
[FT] 단일연도             특징  64 | macro-F1 0.642 정확도 0.642 | 침엽수림 0.67, 활엽수림 0.69, 혼효림 0.57
[CH] 다년(2017~2025)    특징 204 | macro-F1 0.913 정확도 0.924 | 무변화 0.94, 벌채·갱신 0.95, 산림증가 0.84
[CH] 두 시점(2019·2025)만 특징 192 | macro-F1 0.913 정확도 0.923 | 무변화 0.94, 벌채·갱신 0.94, 산림증가 0.85
저장: /content/drive/MyDrive/GEE_Export/ae_class_models


In [5]:
# [12-5] 결과 요약 (클래스별 정밀도·재현율, 검증 표본, 혼동행렬)
for t, r in report.items():
    v = r['다년(2017~2025)']
    print(f"== {t} {r['과업']} | 학습 {r['표본']['학습']} 검증 {r['표본']['검증']}")
    for l, d in v['클래스별'].items():
        print(f"  {l:<6} 표본(학습/검증) {r['클래스별_표본'][l]}  P {d['precision']:.2f} R {d['recall']:.2f} F1 {d['f1-score']:.2f}")
    print('  혼동행렬(행=실제):', v['혼동행렬(행=실제)'])

== LC 산지 피복: 산림·나지·초지·시설지 | 학습 10028 검증 1809
  산림     표본(학습/검증) [4974, 1026]  P 0.89 R 0.95 F1 0.92
  나지     표본(학습/검증) [1874, 258]  P 0.64 R 0.65 F1 0.64
  초지     표본(학습/검증) [524, 63]  P 0.17 R 0.08 F1 0.11
  시설지    표본(학습/검증) [2656, 462]  P 0.87 R 0.78 F1 0.82
  혼동행렬(행=실제): [[978, 29, 0, 19], [44, 167, 19, 28], [20, 30, 5, 8], [59, 37, 6, 360]]
== FT 임상: 침엽수림·활엽수림·혼효림 | 학습 29764 검증 5939
  침엽수림   표본(학습/검증) [10195, 1805]  P 0.66 R 0.69 F1 0.67
  활엽수림   표본(학습/검증) [9931, 2069]  P 0.68 R 0.72 F1 0.70
  혼효림    표본(학습/검증) [9638, 2065]  P 0.59 R 0.54 F1 0.57
  혼동행렬(행=실제): [[1242, 218, 345], [167, 1487, 415], [483, 468, 1114]]
== CH 변화유형 2019→2025: 무변화·벌채·갱신·산림증가 (산지전용 제외) | 학습 11233 검증 1925
  무변화    표본(학습/검증) [5029, 971]  P 0.93 R 0.96 F1 0.94
  벌채·갱신  표본(학습/검증) [3789, 529]  P 0.98 R 0.93 F1 0.95
  산림증가   표본(학습/검증) [2415, 425]  P 0.86 R 0.83 F1 0.84
  혼동행렬(행=실제): [[933, 5, 33], [11, 492, 26], [64, 7, 354]]
